In [66]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import yaml

In [67]:
with open('../config.yaml', 'r') as file:
    config = yaml.safe_load(file)

In [84]:
# import the datasets
final_demo = pd.read_csv(config["input_data"]["file1"])
df_final_experiment_clients =pd.read_csv(config["input_data"]["file2"])
df_final_web_data_pt_1 = pd.read_csv(config["input_data"]["file3"])
df_final_web_data_pt_2 = pd.read_csv(config["input_data"]["file4"])

In [85]:
# check if there is an issue of column type
final_demo.dtypes, df_final_experiment_clients.dtypes, df_final_web_data_pt_1.dtypes

(client_id             int64
 clnt_tenure_yr      float64
 clnt_tenure_mnth    float64
 clnt_age            float64
 gendr                   str
 num_accts           float64
 bal                 float64
 calls_6_mnth        float64
 logons_6_mnth       float64
 dtype: object,
 client_id    int64
 Variation      str
 dtype: object,
 client_id       int64
 visitor_id        str
 visit_id          str
 process_step      str
 date_time         str
 dtype: object)

The dtype shows that the client_id is always an integer so theire is no issue for joining df

In [86]:
# remove rows from df_exeprimental_clients where variation is NA
df_final_experiment_clients = df_final_experiment_clients.dropna(subset=["Variation"])
df_final_experiment_clients.shape 

(50500, 2)

The decrease of number of rows matches the limitation to clients that participated

### Merge process

In [87]:
# combine the web data
concat_webdata = pd.concat([df_final_web_data_pt_1, df_final_web_data_pt_1], axis=0, join="outer")
concat_webdata.shape

(686282, 5)

In [88]:
# merge the demo
merged_demo = concat_webdata.merge(final_demo, on="client_id",how="left")
merged_demo.shape

(686282, 13)

The constant number of rows show the left join was made properly

In [89]:
# merge the experimental_clients
merged_experiment = merged_demo.merge(df_final_experiment_clients, on="client_id",how="left")
merged_experiment.shape

(686282, 14)

The constant number of rows show the left join was made properly

In [90]:
# separate process from final df
df = merged_experiment.copy()

### Fixing steps

In [91]:
# fix : clean the duplicates values
df.drop_duplicates(inplace=True)

In [92]:
# fix: keep clients involved in the AB Testing
df = df.dropna(subset=["Variation"])

In [93]:
# fix: convert date str into a datetime
df['date_time'] = pd.to_datetime(df['date_time'])

Some client_ids were not matching. For example client_id = 6373617 was in df_final_web_data_pt_2 but not in expepriment either demo. This suggests a discrepancies between datasets (client_id in web data but not in demo for example)

Merge data process:
First the 2 web-data files have been concatenated in order to get the biggest dataset of the process, with the prior variables to be investigated.
Second, this dataset merged additional columns from final_demo which is the clients demographics.
Third, the final exprimental client dataset was added in order to see if the client was involved in a test or control group.
Then we assume to keep only data from our 2 samples.

### Check timeframe and potential cuts

In [94]:
# check if the process wac cut the first and last days of the experience
first_d = df['date_time'].min()
last_d = df['date_time'].max()
first_d,last_d

(Timestamp('2017-03-15 00:19:28'), Timestamp('2017-04-30 23:58:00'))

The last date of web data is 30th of April. Therefore, it is within the exprimental timeframe (June)

In [95]:
first_day_sample = df[df['date_time'].dt.date == pd.to_datetime('2017-03-15').date()].sort_values(by=['date_time'], ascending=True).head(5)[['process_step', 'date_time']]
first_day_sample

,process_step,date_time
35293,start,2017-03-15 00:19:28
35292,step_1,2017-03-15 00:20:50
35291,step_2,2017-03-15 00:22:52
35290,step_3,2017-03-15 00:23:47
35289,confirm,2017-03-15 00:26:24


The first user completed the entire process therefore we consider there was no cut of the process due to the timeframe

In [96]:
# Filter df for a specific visit_id
visit_id_to_keep = "100952948_77681780860_858472"

df_filtered = df[df['visit_id'] == visit_id_to_keep].copy()

# Optional: sort by date_time for clarity
df_filtered = df_filtered.sort_values('date_time').reset_index(drop=True)

df_filtered

,client_id,visitor_id,visit_id,process_step,date_time,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,Variation


### Keep first start datetime and last confirm datetime 

In [80]:
# First start
first_start = (
    df[df['process_step'] == 'start']
    .groupby('visit_id', as_index=False)['date_time']
    .min()
    .rename(columns={'date_time': 'first_start_time'})
)

# Last confirm
last_confirm = (
    df[df['process_step'] == 'confirm']
    .groupby('visit_id', as_index=False)['date_time']
    .max()
    .rename(columns={'date_time': 'last_confirm_time'})
)

# Keep only visits that have both a start and a confirm
visit_limits = first_start.merge(last_confirm, on='visit_id', how='inner')

# Attach those limits back to the original data
df = df.merge(visit_limits, on='visit_id', how='inner')

# Keep only rows between first start and last confirm
df = df[
    (df['date_time'] >= df['first_start_time']) &
    (df['date_time'] <= df['last_confirm_time'])
].copy()

# Sort neatly
df = df.sort_values(['visit_id', 'date_time'])
df = df.reset_index(drop=True)

df.head(20)

,client_id,visitor_id,visit_id,process_step,date_time,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,Variation,first_start_time,last_confirm_time
0,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,7.0,88.0,23.5,M,2.0,26436.73,6.0,9.0,Test,2017-04-09 16:20:56,2017-04-09 16:24:58
1,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,7.0,88.0,23.5,M,2.0,26436.73,6.0,9.0,Test,2017-04-09 16:20:56,2017-04-09 16:24:58
2,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,7.0,88.0,23.5,M,2.0,26436.73,6.0,9.0,Test,2017-04-09 16:20:56,2017-04-09 16:24:58
3,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:35,7.0,88.0,23.5,M,2.0,26436.73,6.0,9.0,Test,2017-04-09 16:20:56,2017-04-09 16:24:58
4,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:41,7.0,88.0,23.5,M,2.0,26436.73,6.0,9.0,Test,2017-04-09 16:20:56,2017-04-09 16:24:58
5,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:21:45,7.0,88.0,23.5,M,2.0,26436.73,6.0,9.0,Test,2017-04-09 16:20:56,2017-04-09 16:24:58
6,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:21:59,7.0,88.0,23.5,M,2.0,26436.73,6.0,9.0,Test,2017-04-09 16:20:56,2017-04-09 16:24:58
7,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:22:04,7.0,88.0,23.5,M,2.0,26436.73,6.0,9.0,Test,2017-04-09 16:20:56,2017-04-09 16:24:58
8,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:22:08,7.0,88.0,23.5,M,2.0,26436.73,6.0,9.0,Test,2017-04-09 16:20:56,2017-04-09 16:24:58
9,7338123,612065484_94198474375,100019538_17884295066_43909,step_3,2017-04-09 16:24:01,7.0,88.0,23.5,M,2.0,26436.73,6.0,9.0,Test,2017-04-09 16:20:56,2017-04-09 16:24:58


## Keep only first start_date row when there are sevral consecutive start dates, same process for confirm dates (but last).
#### this allows to keep track of start and confirm dates that are within the visit (and keep track of restarts)

In [81]:
# Sort by visit_id and date_time to ensure chronological order
df = df.sort_values(['visit_id', 'date_time'])

# Compare each start with the previous row in the same visit
df['prev_step'] = df.groupby('visit_id')['process_step'].shift(1)

# Keep rows:
# - all non-start steps, OR
# - first start in a consecutive sequence
df = df[
    (df['process_step'] != 'start') | 
    ((df['process_step'] == 'start') & (df['prev_step'] != 'start'))
]

# Drop helper column
df = df.drop(columns=['prev_step'])

# Reset index for clean DataFrame
df = df.reset_index(drop=True)

In [82]:
# Sort by visit_id and date_time
df = df.sort_values(['visit_id', 'date_time'])

# Compare each confirm with the next row in the same visit
df['next_step'] = df.groupby('visit_id')['process_step'].shift(-1)

# Keep rows:
# - all non-confirm steps, OR
# - last confirm in a consecutive sequence (next_step != confirm)
df = df[
    (df['process_step'] != 'confirm') |
    ((df['process_step'] == 'confirm') & (df['next_step'] != 'confirm'))
]

# Drop helper column
df = df.drop(columns=['next_step'])

# Reset index for a clean DataFrame
df = df.reset_index(drop=True)

In [83]:
# export the dataset
df_main = df.to_csv(config["output_data"]["file5"])